# Informe técnico — ADY1100 EVA2
## Banco Financiero Global · Campañas de depósitos a plazo

**Dataset:** `bank-additional-full.csv` (UCI · campañas 2008–2010).

Este notebook cumple la entrega del **informe técnico en `.ipynb`**: al menos **17 visualizaciones** con justificación del tipo de gráfico, código comentado (Matplotlib / Seaborn) e interpretación en contexto del negocio.


## 0. Configuración y carga de datos

- Se agrega la raíz del proyecto a `sys.path` para importar `src`.
- **`df_raw`**: datos originales (etiquetas legibles para el cliente).
- **`df_proc`**: datos procesados desde `data/processed/bank_cleaned.csv` (correlaciones numéricas ya homogéneas).
- Las figuras se guardan en `outputs/figures/` con prefijos `g01` … `g17`.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Raíz del proyecto: funciona desde notebooks/ o desde la raíz
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.load_data import get_outputs_dir, get_processed_dir, load_bank_data
from src.visualization import (
    plot_barplot_rate,
    plot_boxplot_bivariate,
    plot_correlation_heatmap,
    plot_countplot,
    plot_histogram_kde,
    plot_macro_trends,
    plot_pie_distribution,
    plot_scatter_with_hue,
    plot_target_distribution,
    plot_violin,
)

FIG_DIR = get_outputs_dir() / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Cargar CSV crudo (separador ;)
df_raw = load_bank_data()
proc_path = get_processed_dir() / "bank_cleaned.csv"
if not proc_path.is_file():
    raise FileNotFoundError(
        f"Ejecute primero el notebook de limpieza para generar: {proc_path}"
    )
df_proc = pd.read_csv(proc_path)

print("Filas raw:", df_raw.shape[0], "| Filas procesadas:", df_proc.shape[0])


## 1. Contexto del caso

El **Banco Financiero Global** busca mejorar la conversión en **depósitos a plazo**. Las campañas telefónicas registran cliente, canal, timing y contexto macroeconómico. El informe visual orienta dónde enfocar esfuerzos **antes** de modelos predictivos.

**Integridad:** no utilizamos `duration` como variable explicativa en este informe analítico exploratorio porque se conoce tras la llamada y puede distorsionar conclusiones operativas (alineado al caso de estudio).


## 2. Variable objetivo `y`

**G01 — Barras (`countplot`):** Comparar dos categorías discretas es más legible con barras que con texto tabular.

**G02 — Torta (`pie`):** Con solo dos categorías, la torta comunica rápidamente la **participación porcentual** en reuniones ejecutivas (uso complementario a las barras).


In [ ]:
# G01: frecuencias absolutas de suscripción
plot_target_distribution(df_raw, save_path=FIG_DIR / "g01_countplot_y.png")

# G02: proporción yes/no sobre el total
plot_pie_distribution(df_raw, "y", title="Proporción de suscripción a depósito a plazo", save_path=FIG_DIR / "g02_pie_y.png")


## 3. Perfil demográfico

**G03 — Histograma + KDE:** La edad es continua; el histograma con KDE muestra forma y posibles colas.

**G04 — Barras horizontales:** `job` tiene muchas categorías; orientación horizontal evita solapamiento de etiquetas.

**G05 — Barras con orden:** `education` tiene orden natural de nivel; ordenamos ejes para lectura coherente.

**G06–G07 — Barras con intervalo (`barplot`):** La **tasa media** de `y` por segmento muestra qué grupos convierten más (métrica clave para marketing).


In [ ]:
# G03
plot_histogram_kde(df_raw, "age", title="Distribución de edad del cliente", save_path=FIG_DIR / "g03_hist_age.png")

# G04 — orden por frecuencia ascendente para barras horizontales legibles
job_order = df_raw["job"].value_counts(ascending=True).index.tolist()
plot_countplot(df_raw, "job", orient="h", order=job_order, title="Distribución por tipo de trabajo", save_path=FIG_DIR / "g04_count_job.png")

# G05 — orden aproximado de nivel educativo (solo niveles presentes en los datos)
edu_levels = [
    "illiterate", "basic.4y", "basic.6y", "basic.9y", "high.school",
    "professional.course", "university.degree", "unknown",
]
edu_order = [e for e in edu_levels if e in df_raw["education"].unique()]
plot_countplot(df_raw, "education", order=edu_order, title="Distribución por nivel educativo", save_path=FIG_DIR / "g05_count_education.png")

# G06–G07 tasas de conversión por trabajo y educación
plot_barplot_rate(df_raw, "job", title="Tasa de suscripción por ocupación", save_path=FIG_DIR / "g06_rate_job.png")
plot_barplot_rate(df_raw, "education", order=edu_order, title="Tasa de suscripción por educación", save_path=FIG_DIR / "g07_rate_education.png")


## 4. Comportamiento de la campaña

**G08 — Histograma:** `campaign` suele estar muy sesgado; visualizar la masa ayuda a debatir políticas de número máximo de contactos.

**G09–G11 — Barras agrupadas (`hue=y`):** Comparan simultáneamente volumen y composición entre suscriptores y no suscriptores por canal, mes y resultado previo.


In [ ]:
# G08
plot_histogram_kde(df_raw, "campaign", title="Distribución del número de contactos en la campaña", save_path=FIG_DIR / "g08_hist_campaign.png")

# G09–G11 — hue separa yes/no
month_order = ["mar","apr","may","jun","jul","aug","sep","oct","nov","dec","jan","feb"]
mo = [m for m in month_order if m in df_raw["month"].unique()]
plot_countplot(df_raw, "contact", hue="y", title="Canal de contacto por resultado", save_path=FIG_DIR / "g09_count_contact_y.png")
plot_countplot(df_raw, "month", hue="y", order=mo, title="Mes del contacto por resultado", save_path=FIG_DIR / "g10_count_month_y.png")
plot_countplot(df_raw, "poutcome", hue="y", title="Resultado de campaña anterior por resultado actual", save_path=FIG_DIR / "g11_count_poutcome_y.png")


## 5. Contexto económico

**G12 — Boxplot:** Relaciona una variable macro continua (`euribor3m`) con el resultado binario; muestra mediana y dispersión por grupo.

**G13 — Líneas:** Para series temporales implícitas en el calendario de campaña, el **lineplot** agregado por mes resume tendencias de indicadores macro.


In [ ]:
# G12
plot_boxplot_bivariate(df_raw, "y", "euribor3m", title="Euribor 3m según suscripción", save_path=FIG_DIR / "g12_box_euribor_y.png")

# G13 — promedio por mes del último contacto
macro_cols = ["emp.var.rate", "cons.price.idx", "cons.conf.idx"]
plot_macro_trends(df_raw, macro_cols, group_col="month", save_path=FIG_DIR / "g13_macro_month.png")


## 6. Relaciones multivariadas

**G14 — Heatmap:** Vista rápida de correlaciones entre numéricas en el dataset ya procesado y escalado.

**G15–G16:** Boxplot y violin comparan distribuciones de edad y de campaña entre grupos `y`.

**G17 — Dispersión:** Explora la nube entre dos macro-indicadores coloreada por `y`.


In [ ]:
# G14 — correlaciones sobre datos numéricos procesados
plot_correlation_heatmap(df_proc, title="Mapa de correlaciones (datos procesados)", save_path=FIG_DIR / "g14_heatmap_corr.png")

# G15–G16
plot_boxplot_bivariate(df_raw, "y", "age", title="Edad según suscripción", save_path=FIG_DIR / "g15_box_age_y.png")
plot_violin(df_raw, "y", "campaign", title="Contactos en campaña según suscripción", save_path=FIG_DIR / "g16_violin_campaign_y.png")

# G17
plot_scatter_with_hue(df_raw, "euribor3m", "nr.employed", "y", title="Euribor 3m vs empleo (color=suscripción)", save_path=FIG_DIR / "g17_scatter_macro_y.png")


## 7. Conclusiones para la gerencia

- La clase objetivo está **desbalanceada** (G01–G02): cualquier decisión debe considerar tasas y no solo conteos brutos.
- Segmentos de **trabajo** y **educación** muestran tasas heterogéneas (G06–G07): priorizar microsegmentos mejora ROI.
- **Mes**, **canal** y **poutcome** modifican la composición de éxitos (G09–G11): coordinación operativa y remarketing.
- Variables macro contextualizan la época de crisis/recuperación (G12–G13); la dispersión G17 muestra covariación entre indicadores.
- El heatmap G14 ayuda a detectar **redundancia** entre predictores antes de modelar.

**Ética:** las visualizaciones describen patrones agregados sin identificar personas; se evita usar variables post-contacto como `duration` para guiar políticas predictivas.


## 8. Referencias

- Moro, S., Cortez, P., & Rita, P. (2014). A Data-Driven Approach to Predict the Success of Bank Telemarketing. *Decision Support Systems*, 62, 22–31. https://doi.org/10.1016/j.dss.2014.03.001

- Hunter, J. D. (2007). Matplotlib: A 2D graphics environment. *Computing in Science & Engineering*, 9(3), 90–95.

- Waskom, M. L. (2021). seaborn: statistical data visualization. *Journal of Open Source Software*, 6(60), 3021. https://doi.org/10.21105/joss.03021

- Documentación Matplotlib: https://matplotlib.org/stable/

- Documentación Seaborn: https://seaborn.pydata.org/
